# Gemma-12B -- Climate generalization pilot

Content-domain + chart-type generalization test (Chapter 7 Limitation 2, Chapter 8 item 4,
supervisor item 7). Solar-vs-wind line chart, fabricated data, climate-framed claim -- see
`climate_pilot/generate_climate_stimuli.py` for the stimulus design rationale. Same protocols
as the main study (baseline single-image like/scroll + logprobs, single-image across the 6
`metrics/realistic` engagement scales + logprobs, full 7x7 paired A/B `metrics` grid), pointed
at `climate_pilot/posts/` instead of `benchmarking/`. 25 posts (not 50/100) -- this is a scoped
pilot, not a full replication.

In [1]:
import sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "tokenizers>=0.22.0,<=0.23.0",
    "--user", "-q"], check=True)

print("✅ Done — restart the kernel now")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.16.0.dev0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.23.1 which is incompatible.


✅ Done — restart the kernel now


Restart kernel after running the setup cell above.

In [2]:
!nvidia-smi

Tue Aug 18 18:38:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:C6:00.0 Off |                   On |
| N/A   28C    P0             78W /  700W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [3]:
# --- HF Auth ---
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

# --- Path setup ---
from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

# --- Load Gemma model ---
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/gemma-4-12B-it"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")
device = model.device

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

In [4]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import (
    LIKE_PROMPT_SINGLE, LIKE_PROMPT_YESNO, LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_baseline, run_e1_metrics, run_e1_metrics_paired
)
from e1_utils.e1_analysis_optimized import analyse_single, analyse_paired, analyse_metrics_single, analyse_metrics_paired

# --- Configuration: climate pilot, NOT the main benchmarking/ pool ---
EXPERIMENT_DIR = Path().resolve().parent        # experiments/e1_climate/  -- shared across all 4 climate-pilot models, so they see the identical 25-image sample
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 25

correct_dir = ROOT_DIR / "climate_pilot/posts/correct/PNGs"
incorrect_dir = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs"
all_images = build_paired_sample(correct_dir, incorrect_dir, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# metrics/realistic condition only -- the condition that showed the strongest conformity effect
# in the main study (Section 6.2)
correct_base = ROOT_DIR / "climate_pilot/posts/correct/PNGs/metrics/realistic"
incorrect_base = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs/metrics/realistic"


📋 Loading existing selection from /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/selected_images.json
✅ All selected numbers verified in both correct and incorrect folders.
Selected 25 pairs → 50 images total


In [5]:
from e1_utils.inference_gemma import run_inference_gemma

In [6]:
from e1_utils.e1_optimized import (
    run_e1_baseline_logprobs, run_e1_metrics_logprobs,
    LIKE_CANDIDATES_SINGLE, LIKE_CANDIDATES_YESNO
)

from e1_utils.inference_gemma import run_inference_with_scores_gemma

## Approach 1 -- single image, like/scroll, baseline (0 engagement)

In [7]:
run_e1_baseline(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_baseline.json",
              inference_fn=run_inference_gemma)

✅ 001_correct → scroll
✅ 001_incorrect → scroll
✅ 002_correct → scroll
✅ 002_incorrect → scroll
✅ 003_correct → scroll
✅ 003_incorrect → scroll
✅ 004_correct → scroll
✅ 004_incorrect → scroll
✅ 005_correct → scroll
✅ 005_incorrect → scroll
✅ 006_correct → scroll
✅ 006_incorrect → scroll
✅ 007_correct → scroll
✅ 007_incorrect → scroll
✅ 008_correct → scroll
✅ 008_incorrect → scroll
✅ 009_correct → scroll
✅ 009_incorrect → scroll
✅ 010_correct → scroll
✅ 010_incorrect → scroll
✅ 011_correct → scroll
✅ 011_incorrect → scroll
✅ 012_correct → scroll
✅ 012_incorrect → scroll
✅ 013_correct → scroll
✅ 013_incorrect → scroll
✅ 014_correct → scroll
✅ 014_incorrect → scroll
✅ 015_correct → scroll
✅ 015_incorrect → scroll
✅ 016_correct → scroll
✅ 016_incorrect → scroll
✅ 017_correct → scroll
✅ 017_incorrect → scroll
✅ 018_correct → scroll
✅ 018_incorrect → scroll
✅ 019_correct → scroll
✅ 019_incorrect → scroll
✅ 020_correct → scroll
✅ 020_incorrect → scroll
✅ 021_correct → scroll
✅ 021_incorrect →

In [8]:
run_e1_baseline_logprobs(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_baseline_logprobs.json", score_fn=run_inference_with_scores_gemma)

✅ 001_correct → scroll {'like': {'logprob': -24.375225067138672, 'prob_forced_choice': 0.0009110511944006454}, 'scroll': {'logprob': -17.375225067138672, 'prob_forced_choice': 0.9990889488055994}}
✅ 001_incorrect → scroll {'like': {'logprob': -26.562503814697266, 'prob_forced_choice': 0.0004520923568991944}, 'scroll': {'logprob': -18.861331939697266, 'prob_forced_choice': 0.9995479076431009}}
✅ 002_correct → scroll {'like': {'logprob': -24.992300033569336, 'prob_forced_choice': 0.000631243456875032}, 'scroll': {'logprob': -17.625112533569336, 'prob_forced_choice': 0.9993687565431251}}
✅ 002_incorrect → scroll {'like': {'logprob': -24.523578643798828, 'prob_forced_choice': 0.0008626077580284025}, 'scroll': {'logprob': -17.468891143798828, 'prob_forced_choice': 0.9991373922419715}}
✅ 003_correct → scroll {'like': {'logprob': -24.281475067138672, 'prob_forced_choice': 0.0010005044009099887}, 'scroll': {'logprob': -17.375225067138672, 'prob_forced_choice': 0.99899949559909}}
✅ 003_incorrec

In [9]:
analyse_single(OUTPUT_DIR, "e1_results_baseline.json", like_answer="like")


Single image analysis: e1_results_baseline
=== Summary ===


,metric,value
0,overall_like_rate_%,0.0
1,like_rate_correct_%,0.0
2,like_rate_incorrect_%,0.0


=== Per Image Results ===


,image,variant,prompt,answer
0,001_correct,correct,You are shown a social media post.\nYou can ei...,scroll
1,001_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
2,002_correct,correct,You are shown a social media post.\nYou can ei...,scroll
3,002_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
4,003_correct,correct,You are shown a social media post.\nYou can ei...,scroll
5,003_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
6,004_correct,correct,You are shown a social media post.\nYou can ei...,scroll
7,004_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
8,005_correct,correct,You are shown a social media post.\nYou can ei...,scroll
9,005_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/gemma4-12b/outputs/e1_analysis_baseline.csv


## Approach 1 variant -- single image, like/scroll, across the 6 `metrics/realistic` engagement scales

In [10]:
run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_metrics.json",
              inference_fn=run_inference_gemma)

✅ 001_correct_10 → scroll
✅ 001_incorrect_10 → scroll
✅ 002_correct_10 → scroll
✅ 002_incorrect_10 → scroll
✅ 003_correct_10 → scroll
✅ 003_incorrect_10 → scroll
✅ 004_correct_10 → scroll
✅ 004_incorrect_10 → scroll
✅ 005_correct_10 → scroll
✅ 005_incorrect_10 → scroll
✅ 006_correct_10 → scroll
✅ 006_incorrect_10 → scroll
✅ 007_correct_10 → scroll
✅ 007_incorrect_10 → scroll
✅ 008_correct_10 → scroll
✅ 008_incorrect_10 → scroll
✅ 009_correct_10 → scroll
✅ 009_incorrect_10 → scroll
✅ 010_correct_10 → scroll
✅ 010_incorrect_10 → scroll
✅ 011_correct_10 → scroll
✅ 011_incorrect_10 → scroll
✅ 012_correct_10 → scroll
✅ 012_incorrect_10 → scroll
✅ 013_correct_10 → scroll
✅ 013_incorrect_10 → scroll
✅ 014_correct_10 → scroll
✅ 014_incorrect_10 → scroll
✅ 015_correct_10 → scroll
✅ 015_incorrect_10 → scroll
✅ 016_correct_10 → scroll
✅ 016_incorrect_10 → scroll
✅ 017_correct_10 → scroll
✅ 017_incorrect_10 → scroll
✅ 018_correct_10 → scroll
✅ 018_incorrect_10 → scroll
✅ 019_correct_10 → scroll
✅ 

In [11]:
run_e1_metrics_logprobs(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_metrics_logprobs.json", score_fn=run_inference_with_scores_gemma)

✅ 001_correct_10 → scroll {'like': {'logprob': -24.26585578918457, 'prob_forced_choice': 0.0011513995656772138}, 'scroll': {'logprob': -17.50023078918457, 'prob_forced_choice': 0.9988486004343227}}
✅ 001_incorrect_10 → scroll {'like': {'logprob': -26.968751907348633, 'prob_forced_choice': 0.00033274128340681067}, 'scroll': {'logprob': -18.960939407348633, 'prob_forced_choice': 0.9996672587165932}}
✅ 002_correct_10 → scroll {'like': {'logprob': -24.875093460083008, 'prob_forced_choice': 0.0008295891883591028}, 'scroll': {'logprob': -17.781343460083008, 'prob_forced_choice': 0.9991704108116409}}
✅ 002_incorrect_10 → scroll {'like': {'logprob': -25.187585830688477, 'prob_forced_choice': 0.0007321812616608614}, 'scroll': {'logprob': -17.968835830688477, 'prob_forced_choice': 0.9992678187383391}}
✅ 003_correct_10 → scroll {'like': {'logprob': -24.687625885009766, 'prob_forced_choice': 0.0008040859356292355}, 'scroll': {'logprob': -17.562625885009766, 'prob_forced_choice': 0.9991959140643708

In [12]:
analyse_metrics_single(OUTPUT_DIR, "e1_results_metrics.json", like_answer="like")


Metrics single image analysis: e1_results_metrics
=== Overall Summary ===


,metric,value
0,overall_like_rate_%,0.0
1,like_rate_correct_%,0.0
2,like_rate_incorrect_%,0.0


=== Rate per Scale Value ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:116: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_scale = df.groupby("scale_value").apply(lambda g: pd.Series({


,scale_value,total_images,overall_like_rate_%,like_rate_correct_%,like_rate_incorrect_%
0,10,50.0,0.0,0.0,0.0
1,100,50.0,0.0,0.0,0.0
2,1000,50.0,0.0,0.0,0.0
3,10000,50.0,0.0,0.0,0.0
4,100000,50.0,0.0,0.0,0.0
5,1000000,50.0,0.0,0.0,0.0


=== Per Image Results ===


,image,num,variant,scale_value,prompt,answer
0,001_correct_10,001,correct,10,You are shown a social media post.\nYou can ei...,scroll
2,002_correct_10,002,correct,10,You are shown a social media post.\nYou can ei...,scroll
4,003_correct_10,003,correct,10,You are shown a social media post.\nYou can ei...,scroll
6,004_correct_10,004,correct,10,You are shown a social media post.\nYou can ei...,scroll
8,005_correct_10,005,correct,10,You are shown a social media post.\nYou can ei...,scroll
...,...,...,...,...,...,...
291,021_incorrect_1000000,021,incorrect,1000000,You are shown a social media post.\nYou can ei...,scroll
293,022_incorrect_1000000,022,incorrect,1000000,You are shown a social media post.\nYou can ei...,scroll
295,023_incorrect_1000000,023,incorrect,1000000,You are shown a social media post.\nYou can ei...,scroll
297,024_incorrect_1000000,024,incorrect,1000000,You are shown a social media post.\nYou can ei...,scroll


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/gemma4-12b/outputs/e1_analysis_metrics.csv


## Approach 2 -- paired A/B forced choice, full 7x7 `metrics/realistic` disparity grid

In [13]:
import time
start = time.time()
run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
              prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_paired.json",
              inference_fn=run_inference_gemma)
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min")

📋 Resuming — 1225 pairs already processed.
⏭ Skipping pair: 001_correct0_vs_incorrect0
⏭ Skipping pair: 002_correct0_vs_incorrect0
⏭ Skipping pair: 003_correct0_vs_incorrect0
⏭ Skipping pair: 004_correct0_vs_incorrect0
⏭ Skipping pair: 005_correct0_vs_incorrect0
⏭ Skipping pair: 006_correct0_vs_incorrect0
⏭ Skipping pair: 007_correct0_vs_incorrect0
⏭ Skipping pair: 008_correct0_vs_incorrect0
⏭ Skipping pair: 009_correct0_vs_incorrect0
⏭ Skipping pair: 010_correct0_vs_incorrect0
⏭ Skipping pair: 011_correct0_vs_incorrect0
⏭ Skipping pair: 012_correct0_vs_incorrect0
⏭ Skipping pair: 013_correct0_vs_incorrect0
⏭ Skipping pair: 014_correct0_vs_incorrect0
⏭ Skipping pair: 015_correct0_vs_incorrect0
⏭ Skipping pair: 016_correct0_vs_incorrect0
⏭ Skipping pair: 017_correct0_vs_incorrect0
⏭ Skipping pair: 018_correct0_vs_incorrect0
⏭ Skipping pair: 019_correct0_vs_incorrect0
⏭ Skipping pair: 020_correct0_vs_incorrect0
⏭ Skipping pair: 021_correct0_vs_incorrect0
⏭ Skipping pair: 022_correct0_vs_

In [14]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_metrics_paired.json")


Metrics paired A/B analysis: e1_results_metrics_paired
=== Overall Summary ===


,metric,value
0,overall_liked_correct_%,52.33
1,overall_liked_incorrect_%,47.51
2,invalid_answer_%,0.16


=== Liked Correct Rate per Scale Pair ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:163: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_pair = df.groupby("pair").apply(lambda g: pd.Series({


,pair,total_pairs,liked_correct_%,liked_incorrect_%,invalid_%
0,0_vs_0,25.0,80.0,16.0,4.0
1,0_vs_10,25.0,96.0,4.0,0.0
2,0_vs_100,25.0,100.0,0.0,0.0
3,0_vs_1000,25.0,92.0,8.0,0.0
4,0_vs_10000,25.0,100.0,0.0,0.0
5,0_vs_100000,25.0,100.0,0.0,0.0
6,0_vs_1000000,25.0,100.0,0.0,0.0
7,10_vs_0,25.0,24.0,76.0,0.0
8,10_vs_10,25.0,64.0,36.0,0.0
9,10_vs_100,25.0,60.0,40.0,0.0


=== Per Pair Results ===


,image,num,correct_scale,incorrect_scale,post_a_variant,post_b_variant,prompt,answer,liked_variant,pair
0,001_correct0_vs_incorrect0,001,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,0_vs_0
1,002_correct0_vs_incorrect0,002,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
2,003_correct0_vs_incorrect0,003,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
3,004_correct0_vs_incorrect0,004,0,0,incorrect,correct,"You are shown two social media posts, Post A a...",A,incorrect,0_vs_0
4,005_correct0_vs_incorrect0,005,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
...,...,...,...,...,...,...,...,...,...,...
1220,021_correct1000000_vs_incorrect1000000,021,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",A,incorrect,1000000_vs_1000000
1221,022_correct1000000_vs_incorrect1000000,022,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",A,incorrect,1000000_vs_1000000
1222,023_correct1000000_vs_incorrect1000000,023,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct,1000000_vs_1000000
1223,024_correct1000000_vs_incorrect1000000,024,1000000,1000000,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,1000000_vs_1000000


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/gemma4-12b/outputs/e1_analysis_metrics_paired.csv
